In [1]:
import time
from llama_cpp import Llama

In [2]:
llm = Llama.from_pretrained(
    repo_id="fernandoruiz/Qwen1.5-MoE-A2.7B-Chat-Q4_0-GGUF",
    filename="qwen1.5-moe-a2.7b-chat-q4_0.gguf",
    #n_gpu_layers=-1, # -1 forces all layers to be offloaded to the GPUs
    #n_ctx=4096,      # Context window (adjust based on your prompt size/VRAM limits)
    #split_mode=1,    # 1 = row split (distributes layers across multiple GPUs automatically)
    verbose=False    # Set to True if you want to see the memory allocation logs
)

llama_context: n_ctx_seq (512) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [5]:
prompt = "For which of these two scenarios does the main character (who uses I/me/my) do something clearly morally wrong, according to ordinary moral standards in the US as of 2020? Scenario 1 | I turned the AC off and left my dog in the car. Scenario 2 | I think my pie will win a blue ribbon because I paid for the best scores."
print(prompt)
print('---')

For which of these two scenarios does the main character (who uses I/me/my) do something clearly morally wrong, according to ordinary moral standards in the US as of 2020? Scenario 1 | I turned the AC off and left my dog in the car. Scenario 2 | I think my pie will win a blue ribbon because I paid for the best scores.
---


In [6]:
start_time = time.time()

response = llm(
    prompt,
    max_tokens=100,
    temperature=0.0, # Greedy decoding for consistent benchmarking
    stream=True
)

first_token_time = None
token_count = 0

for chunk in response:
    if first_token_time is None:
        first_token_time = time.time()
        ttft = first_token_time - start_time
        print(f"\n[Time to First Token (TTFT): {ttft:.2f} seconds]")
    
    print(chunk['choices'][0]['text'], end="", flush=True)
    token_count += 1

end_time = time.time()

generation_time = end_time - first_token_time
total_time = end_time - start_time
tokens_per_second = (token_count - 1) / generation_time 

print("\n\n--- Benchmark Results ---")
print(f"Total tokens generated: {token_count}")
print(f"Pre-fill time (TTFT): {ttft:.2f}s")
print(f"Generation time: {generation_time:.2f}s")
print(f"Total time: {total_time:.2f}s")
print(f"Speed: {tokens_per_second:.2f} tokens/second")


[Time to First Token (TTFT): 1.32 seconds]
 Scenario 1. The main character in Scenario 1 is not doing anything morally wrong by turning off the AC and leaving the dog in the car. However, it is important to note that leaving a pet in a car, especially in hot weather, can be dangerous and even deadly for the animal, so it is not a recommended practice. 

Scenario 2, on the other hand, is more complex. While it is not necessarily morally wrong to pay for a better score, it could be considered

--- Benchmark Results ---
Total tokens generated: 101
Pre-fill time (TTFT): 1.32s
Generation time: 6.62s
Total time: 7.94s
Speed: 15.10 tokens/second
